In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# ============================================================
# Directional Connectedness (TO / FROM)
# ------------------------------------------------------------
# Input files in ./
#   - ./gfevd_all.npy
#   - ./tvpvar_effective_dates.csv
#   - ./tvpvar_var_names.csv
#
# Output files in ./result
#   - directional_time.csv
#   - directional_mean.csv
#   - to_series.png
#   - from_series.png
#   - to_series_interactive.html
#   - from_series_interactive.html
# ============================================================

# =========================
# Config
# =========================
BASE_DIR = Path("./")
RESULT_DIR = BASE_DIR / "result"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

GFEVD_FILE = BASE_DIR / "gfevd_all.npy"
DATES_FILE = BASE_DIR / "tvpvar_effective_dates.csv"
VARNAMES_FILE = BASE_DIR / "tvpvar_var_names.csv"

OUT_TIME = RESULT_DIR / "directional_time.csv"
OUT_MEAN = RESULT_DIR / "directional_mean.csv"
OUT_TO_PNG = RESULT_DIR / "to_series.png"
OUT_FROM_PNG = RESULT_DIR / "from_series.png"
OUT_TO_HTML = RESULT_DIR / "to_series_interactive.html"
OUT_FROM_HTML = RESULT_DIR / "from_series_interactive.html"

DEFAULT_VAR_NAMES = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "dlog_OIL",
    "dlog_GAS",
    "d_UST10Y",
    "d_VIX",
]

ROW_SUM_TOL = 1e-6
FORCE_ROW_NORMALIZE = True
SCALE_TO_PERCENT = True


# =========================
# Helpers
# =========================
def load_var_names(n):
    if VARNAMES_FILE.exists():
        df_names = pd.read_csv(VARNAMES_FILE)

        if "var_name" not in df_names.columns:
            raise ValueError("tvpvar_var_names.csv must contain 'var_name' column.")

        var_names = df_names["var_name"].astype(str).tolist()

        if len(var_names) != n:
            raise ValueError(
                f"var_names length({len(var_names)}) != GFEVD dimension({n})"
            )

        return var_names

    if len(DEFAULT_VAR_NAMES) != n:
        raise ValueError(
            f"DEFAULT_VAR_NAMES length({len(DEFAULT_VAR_NAMES)}) != GFEVD dimension({n})"
        )

    return DEFAULT_VAR_NAMES


def load_dates(target_len):
    if DATES_FILE.exists():
        df_dates = pd.read_csv(DATES_FILE)

        if "Date" not in df_dates.columns:
            raise ValueError("tvpvar_effective_dates.csv must contain 'Date' column.")

        dates = pd.to_datetime(df_dates["Date"], errors="coerce")

        if dates.isna().any():
            raise ValueError("Invalid Date values found in tvpvar_effective_dates.csv")

        if len(dates) != target_len:
            raise ValueError(
                f"Date length mismatch: len(dates)={len(dates)} != T_eff={target_len}"
            )

        return dates.reset_index(drop=True)

    return None


def row_normalize_theta(theta):
    theta = np.asarray(theta, dtype=float)
    row_sum = theta.sum(axis=1, keepdims=True)

    out = theta.copy()
    valid_rows = np.isfinite(row_sum[:, 0]) & (row_sum[:, 0] != 0)

    out[valid_rows] = out[valid_rows] / row_sum[valid_rows]
    out[~valid_rows] = np.nan

    return out


def compute_to_from(theta):
    """
    theta: (N, N)
    row = response variable
    col = shock source

    FROM_i = sum_j theta[i, j] - theta[i, i]
    TO_i   = sum_j theta[j, i] - theta[i, i]
    """
    diag = np.diag(theta)

    from_ = theta.sum(axis=1) - diag
    to_ = theta.sum(axis=0) - diag

    return to_, from_


# =========================
# Load
# =========================
if not GFEVD_FILE.exists():
    raise FileNotFoundError(f"Not found: {GFEVD_FILE}")

gfevd_all = np.load(GFEVD_FILE)

if gfevd_all.ndim != 3:
    raise ValueError(f"gfevd_all must be 3D, got shape={gfevd_all.shape}")

T_eff, N, N2 = gfevd_all.shape

if N != N2:
    raise ValueError(f"Last two dims must be square, got shape={gfevd_all.shape}")

if not np.all(np.isfinite(gfevd_all)):
    raise ValueError("gfevd_all contains non-finite values.")

var_names = load_var_names(N)
dates = load_dates(T_eff)

print(f"[INFO] Loaded GFEVD: shape={gfevd_all.shape}")
print(f"[INFO] Variables: {var_names}")
print(f"[INFO] Dates: {'loaded' if dates is not None else 'not found -> using t index'}")

# =========================
# Check / Normalize
# =========================
gfevd_proc = gfevd_all.copy()

row_sum_check = gfevd_proc.sum(axis=2)
max_row_err = np.max(np.abs(row_sum_check - 1.0))

print(f"[INFO] Max row-sum error before normalization: {max_row_err:.8f}")

if FORCE_ROW_NORMALIZE and max_row_err > ROW_SUM_TOL:
    print("[INFO] Applying row normalization to all theta_t")

    for t in range(T_eff):
        gfevd_proc[t] = row_normalize_theta(gfevd_proc[t])

row_sum_check_after = gfevd_proc.sum(axis=2)
max_row_err_after = np.nanmax(np.abs(row_sum_check_after - 1.0))

print(f"[INFO] Max row-sum error after normalization: {max_row_err_after:.8f}")

# =========================
# Time-varying TO / FROM
# =========================
records = []

scale = 100.0 if SCALE_TO_PERCENT else 1.0

for t in range(T_eff):
    theta = gfevd_proc[t]

    if np.isnan(theta).any():
        continue

    to_, from_ = compute_to_from(theta)

    row = {"t": t}

    if dates is not None:
        row["Date"] = dates.iloc[t]

    for i, name in enumerate(var_names):
        row[f"TO_{name}"] = to_[i] * scale
        row[f"FROM_{name}"] = from_[i] * scale

    records.append(row)

df_time = pd.DataFrame(records)

if df_time.empty:
    raise RuntimeError("No valid directional connectedness rows were created.")

# =========================
# Mean TO / FROM
# =========================
rows_mean = []

for name in var_names:
    to_mean = df_time[f"TO_{name}"].mean()
    from_mean = df_time[f"FROM_{name}"].mean()

    rows_mean.append({
        "Variable": name,
        "TO_mean": to_mean,
        "FROM_mean": from_mean,
        "NET_mean": to_mean - from_mean,
    })

df_mean = pd.DataFrame(rows_mean)

# =========================
# Save CSV
# =========================
df_time.to_csv(OUT_TIME, index=False, encoding="utf-8-sig")
df_mean.to_csv(OUT_MEAN, index=False, encoding="utf-8-sig")

print("\n[INFO] Saved CSV:")
print(f"  - {OUT_TIME}")
print(f"  - {OUT_MEAN}")

print("\n[INFO] Mean Directional Connectedness")
print(df_mean.to_string(index=False))

# =========================
# X-axis
# =========================
if "Date" in df_time.columns:
    df_time["Date"] = pd.to_datetime(df_time["Date"])
    x = df_time["Date"]
    x_label = "Date"
else:
    x = df_time["t"]
    x_label = "t"

y_label = "Contribution (%)" if SCALE_TO_PERCENT else "Contribution"

# =========================
# Matplotlib Save: TO
# =========================
plt.figure(figsize=(14, 6))

for name in var_names:
    plt.plot(x, df_time[f"TO_{name}"], label=name, linewidth=1.2)

plt.title("Directional TO (Outgoing Spillover)")
plt.xlabel(x_label)
plt.ylabel(y_label)
plt.legend(ncol=3, fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_TO_PNG, dpi=300, bbox_inches="tight")
plt.close()

# =========================
# Matplotlib Save: FROM
# =========================
plt.figure(figsize=(14, 6))

for name in var_names:
    plt.plot(x, df_time[f"FROM_{name}"], label=name, linewidth=1.2)

plt.title("Directional FROM (Incoming Spillover)")
plt.xlabel(x_label)
plt.ylabel(y_label)
plt.legend(ncol=3, fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_FROM_PNG, dpi=300, bbox_inches="tight")
plt.close()

print("\n[INFO] Saved PNG:")
print(f"  - {OUT_TO_PNG}")
print(f"  - {OUT_FROM_PNG}")

# =========================
# Plotly Interactive: TO
# =========================
fig_to = go.Figure()

for name in var_names:
    fig_to.add_trace(
        go.Scatter(
            x=x,
            y=df_time[f"TO_{name}"],
            mode="lines",
            name=name,
        )
    )

fig_to.update_layout(
    title="Interactive Directional TO (Outgoing Spillover)",
    xaxis_title=x_label,
    yaxis_title=y_label,
    height=650,
    hovermode="x unified",
    template="plotly_white",
)

fig_to.write_html(str(OUT_TO_HTML))
fig_to.show()

# =========================
# Plotly Interactive: FROM
# =========================
fig_from = go.Figure()

for name in var_names:
    fig_from.add_trace(
        go.Scatter(
            x=x,
            y=df_time[f"FROM_{name}"],
            mode="lines",
            name=name,
        )
    )

fig_from.update_layout(
    title="Interactive Directional FROM (Incoming Spillover)",
    xaxis_title=x_label,
    yaxis_title=y_label,
    height=650,
    hovermode="x unified",
    template="plotly_white",
)

fig_from.write_html(str(OUT_FROM_HTML))
fig_from.show()

print("\n[INFO] Saved HTML:")
print(f"  - {OUT_TO_HTML}")
print(f"  - {OUT_FROM_HTML}")

print("\nDone.")

[INFO] Loaded GFEVD: shape=(1031, 9, 9)
[INFO] Variables: ['dlog_SOLVPN', 'dlog_COPPER', 'dlog_GOLD', 'dlog_SILVER', 'dlog_DXY', 'dlog_OIL', 'dlog_GAS', 'd_UST10Y', 'd_VIX']
[INFO] Dates: not found -> using t index
[INFO] Max row-sum error before normalization: 0.00000000
[INFO] Max row-sum error after normalization: 0.00000000

[INFO] Saved CSV:
  - result\directional_time.csv
  - result\directional_mean.csv

[INFO] Mean Directional Connectedness
   Variable   TO_mean  FROM_mean   NET_mean
dlog_SOLVPN 69.173563  59.321995   9.851569
dlog_COPPER 54.652962  57.045396  -2.392434
  dlog_GOLD 68.148944  63.266614   4.882330
dlog_SILVER 69.706401  62.770500   6.935901
   dlog_DXY 60.701414  60.315292   0.386122
   dlog_OIL 40.468117  50.700929 -10.232812
   dlog_GAS 32.442555  42.263584  -9.821029
   d_UST10Y 50.832055  55.584546  -4.752491
      d_VIX 60.111719  54.968874   5.142845

[INFO] Saved PNG:
  - result\to_series.png
  - result\from_series.png



[INFO] Saved HTML:
  - result\to_series_interactive.html
  - result\from_series_interactive.html

Done.
